In [47]:
import os
import json
import requests
import getpass
import pandas as pd
from bs4 import BeautifulSoup
import re
import openai
import asyncio
from urllib.parse import urlparse
from recommendation_research_llm_handler.llm_client import LLMHandler

# --- 1. KONFIGURACE ---

In [28]:
os.environ['SELLMA_API_KEY'] = getpass.getpass("Api klíč pro LLM proxy:")

Api klíč pro LLM proxy: ········


In [110]:
# --- KONFIGURACE PIPELINE ---
NUM_TOPICS_TO_GENERATE = 10   # Kolik různých témat má LLM vymyslet
URLS_PER_TOPIC = 30          # Kolik URL hledat pro každé téma
LLM_TEMP = 0.1                # Teplota pro stabilní JSON výstupy

In [130]:
# temperature still subject to tuning
llm_client = LLMHandler(api_key=os.getenv('SELLMA_API_KEY'),
                              model="gpt-4.1",
                              temperature=0.1)

# --- 2. POMOCNÉ FUNKCE ---

In [111]:
def extract_content_from_response(res):
    if isinstance(res, str): return res
    if isinstance(res, dict): return res.get('text') or str(res)
    if hasattr(res, 'choices'): 
        try: return res.choices[0].message.content
        except: pass
    if hasattr(res, 'content'): return res.content
    return str(res)

In [112]:
def extract_json_list(text):
    if not text: return []
    try:
        match = re.search(r'\[.*\]', text, re.DOTALL)
        if match: return json.loads(match.group(0))
    except: pass
    return []

In [92]:
def clean_json_response(text):
    text = re.sub(r'```json\n?', '', text)
    text = re.sub(r'```', '', text)
    return text.strip()

In [5]:
def get_page_content(url):
    """Stáhne HTML a vrátí čistý text (synchronní request je pro prototyp OK)"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (compatible; HobbyBot/1.0)'}
        response = requests.get(url, headers=headers, timeout=8)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            for script in soup(["script", "style", "nav", "footer"]):
                script.extract()
            text = soup.get_text(separator=' ').strip()
            text = re.sub(r'\s+', ' ', text)
            return True, text[:3500] # Limit znaků pro kontext okna
        else:
            return False, f"Status Code: {response.status_code}"
    except Exception as e:
        return False, f"Chyba: {str(e)}"

In [114]:
def analyze_ad_density(html_content):
    # Technická metrika pro LLM (počet skriptů)
    ad_keywords = ["adsbygoogle", "sklik", "etarget", "adform", "doubleclick", "google_ads", "banner", "smartadserver"]
    raw_count = 0
    for kw in ad_keywords: raw_count += html_content.count(kw)
    return min(10, 1 + (raw_count // 2))

In [115]:
def get_page_content_smart(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    try:
        response = requests.get(url, headers=headers, timeout=8)
        # Fallback na root při 404
        if response.status_code == 404:
            root_url = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
            if root_url != url.rstrip('/'):
                response = requests.get(root_url, headers=headers, timeout=8)
                url = root_url # Update URL

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # 1. Změna: MAŽEME JEN TO NEJNUTNĚJŠÍ. Necháváme <nav>, <header>, <footer>
            for tag in soup(["script", "style", "svg", "iframe", "noscript", "form"]): 
                tag.extract()
            
            text = soup.get_text(separator=' ').strip()
            text = re.sub(r'\s+', ' ', text)
            
            # Spočítáme reklamy pro kontext
            ad_score = analyze_ad_density(response.text)
            
            return True, url, text[:4000], f"AdScore: {ad_score}/10", ad_score
            
        return False, url, f"Status: {response.status_code}", "N/A", 0
    except Exception as e:
        return False, url, str(e), "N/A", 0

# --- LLM AGENTI ---

In [132]:
async def generate_niche_topics(count):
    print(f"🧠 Brainstorming {count} zájmových témat...")
    prompt = f"""
    # ROLE
    Jsi Senior Content Strategist a expert na analýzu trendů. Tvým úkolem je identifikovat specifické "High-Engagement" hobby niky (výklenky) pro český trh.

    # ÚKOL
    Vygeneruj seznam {count} specifických hobby témat v češtině.
    Musí se jednat o tzv. "Hard Hobbies" – tedy zájmy, které jdou do hloubky a mají silnou fanouškovskou základnu.
    
    # DEFINICE TÉMATU (KRITÉRIA VÝBĚRU)
    Neomezuj se na konkrétní kategorie. Vybírej témata napříč všemi obory (technika, příroda, řemeslo, umění, sběratelství), která splňují tyto vlastnosti:
    1. **Odbornost:** Vyžadují učení a "skill" (nejde o pasivní zábavu).
    2. **Vybavení (Gear):** Existuje kolem nich trh se specifickými nástroji nebo hardwarem.
    3. **Komunita:** Téma má prokazatelnou online komunitu (fóra, skupiny, YouTube kanály), která řeší technické detaily.
    4. **Search Volume:** Téma je reálně hledané (lidé hledají návody, recenze, postupy).
    
    # KONTROLA KVALITY (ANTI-HALLUCINATION)
    - Vyhni se obecným termínům (NE: "Vaření", "Cestování", "Sport").
    - Buď konkrétní (ANO: "Fermentace zeleniny", "Ultralight treking", "Simracing").
    - Používej zavedenou českou terminologii nebo v ČR běžně používané anglicismy (např. "Scrapbooking").
    
    # FORMÁT VÝSTUPU
    Vrať POUZE validní JSON pole stringů. Žádný markdown, žádný úvodní text.
    
    ["Téma 1", "Téma 2", ... "Téma {count}"]
    """
    try:
        res = await llm_client.single_response(prompt=prompt, temperature=0.7, output_metadata=True)
        return extract_json_list(extract_content_from_response(res))
    except: return []

In [121]:
async def generate_seed_urls(topic, count):
    print(f"   ↳ Hledám zdroje pro: '{topic}'...")
    prompt = f"""
    Jsi expert na vyhledávání blogů. Vypiš {count} URL pro téma: "{topic}".
    
    PODMÍNKY:
    1. Musí to být BLOGY nebo MAGAZÍNY (obsahové weby).
    2. IGNORUJ obecné portály (Dama, Idnes, Novinky).
    3. IGNORUJ weby, které nemají sekci článků (jen firemní vizitky).
    
    VÝSTUP JSON POLE: ["url1", "url2"]
    """
    try:
        res = await llm_client.single_response(prompt=prompt, temperature=LLM_TEMP, output_metadata=True)
        return extract_json_list(extract_content_from_response(res))
    except: return []

In [122]:
# --- C. AUDITOR OBSAHU ---
async def audit_url_content(url, text_content, ad_info, topic):
    # ZMĚNA: Předáváme TÉMA a kontrolujeme zaměření celého webu
    prompt = f"""
    Jsi striktní auditor kvality. Rozhodni, zda web {url} patří do datasetu pro téma: "{topic}".
    
    KONTEXT:
    Hledáme pouze úzce specializované "niche" weby.
    Metadata: {ad_info}
    
    OBSAH WEBU:
    {text_content[:3000]}...
    
    ROZHODOVACÍ STROM:
    1. JE TO OBECNÝ PORTÁL? (Píše se tam o všem možném - recepty, horoskopy, zprávy, celebrity?) 
       -> ZAMITNUTO (Důvod: Generalist/Mainstream)
       -> Příklad zamítnutí: dama.cz, toprecepty.cz, kudyznudy.cz, novinky.cz
       
    2. JE TO E-SHOP / KATALOG FIREM? (Hlavní cíl je prodej, ne články?)
       -> ZAMITNUTO (Důvod: Eshop/Corporate)
       
    3. JE WEB 100% ZAMĚŘENÝ NA TÉMA "{topic}"?
       -> Pokud ANO a má články -> SCHVALENO (Type: Blog/Magazin)
    
    VÝSTUP JSON:
    {{
        "verdict": "SCHVALENO" / "ZAMITNUTO",
        "type": "Blog" / "Generalist" / "Eshop" / "Forum",
        "reason": "Vysvětlení"
    }}
    """
    try:
        res = await llm_client.single_response(prompt=prompt, temperature=LLM_TEMP, output_metadata=True)
        text = extract_content_from_response(res)
        match = re.search(r'\{.*\}', text, re.DOTALL)
        return json.loads(match.group(0)) if match else {"verdict": "ERROR", "reason": "No JSON"}
    except Exception as e:
        return {"verdict": "ERROR", "reason": str(e)}

# --- MAIN EXECUTION LOOP ---

In [133]:
# 1. Vygenerování témat
topics = await generate_niche_topics(NUM_TOPICS_TO_GENERATE)
print(f"📝 Témata: {', '.join(topics)}\n")

all_results = []
processed_domains = set()

🧠 Brainstorming 10 zájmových témat...
📝 Témata: Simracing, Fermentace zeleniny, Ultralight treking, Airsoft, Akvaristika s krevetami, Domácí pivovarnictví, FPV drony, Modelová železnice, Výroba nožů (nožířství), Sběratelství LEGO



In [134]:
# 2. Iterace přes témata
for i, topic in enumerate(topics, 1):
    print(f"--- {topic.upper()} ---")
    seed_urls = await generate_seed_urls(topic, URLS_PER_TOPIC)
    unique_inputs = list(set(seed_urls))
    
    for url in unique_inputs:
        if any(x in url for x in ["forum", "diskuse", "viewtopic", "cart", "product", "facebook", "instagram"]):
            continue

        domain = urlparse(url).netloc.replace("www.", "")
        if domain in processed_domains: continue
        
        print(f"   🔎 {url} ... ", end="")

        success, final_url, content, ad_info_str, ad_score_int = get_page_content_smart(url)

        if not success:
            print(f"❌ Nedostupné")
            continue
        
        final_domain = urlparse(final_url).netloc.replace("www.", "")
        processed_domains.add(final_domain)

        # !!! ZMĚNA ZDE: Předáváme 'topic' do auditora !!!
        audit = await audit_url_content(final_url, content, ad_info_str, topic)
        
        icon = "✅" if audit.get('verdict') == "SCHVALENO" else "⛔"
        # Vypíšeme i důvod pro zamítnutí, abychom viděli, jestli filtr funguje
        reason_short = audit.get('reason', '')[:40] 
        print(f"{icon} {audit.get('type')} ({reason_short}...)")

        all_results.append({
            "topic": topic,
            "url": final_url,
            "domain": final_domain,
            "status": audit.get('verdict'),
            "type": audit.get('type'),
            "ad_saturation": ad_score_int,
            "reason": audit.get('reason')
        })
    print("")

print("🏁 Hotovo.")

--- SIMRACING ---
   ↳ Hledám zdroje pro: 'Simracing'...
   🔎 https://www.simracing.cz/blog ... ❌ Nedostupné
   🔎 https://www.simracingpc.com ... ❌ Nedostupné
   🔎 https://www.simracinglife.com ... ⛔ Eshop (Web simracinglife.com je primárně e-shop...)
   🔎 https://www.simracinggarage.com ... ✅ Blog (Web je úzce specializovaný na téma simra...)
   🔎 https://www.simracingreview.com ... ❌ Nedostupné
   🔎 https://www.simracinglinks.com ... ⛔ Generalist (Web simracinglinks.com není úzce zaměřen...)
   🔎 https://www.simracing-news.com ... ✅ Blog (Web simracing-news.com je úzce specializ...)
   🔎 https://www.simracingdeal.com/blog ... ⛔ Eshop (Web zobrazuje pouze ověření bota a není ...)
   🔎 https://www.simracingwiki.com ... ❌ Nedostupné
   🔎 https://simraceblog.com ... ❌ Nedostupné
   🔎 https://www.simracing247.com ... ❌ Nedostupné
   🔎 https://www.simracingworld.com ... ⛔ Generalist (Web simracingworld.com je pouze zaparkov...)
   🔎 https://www.simracingcoach.com/blog ... ⛔ Eshop (Web simr

# Zobrazení výsledků

In [136]:
# --- KONFIGURACE FILTRU ---
MAX_AD_SATURATION = 4  # Weby s vyšším skóre budou zahozeny

if all_results:
    df = pd.DataFrame(all_results)
    
    # 1. Základní filtr: Jen schválené LLM auditorem
    df_approved = df[df['status'] == 'SCHVALENO'].copy()
    
    # 2. Filtr reklam: Vyhodíme weby přesycené reklamou
    # Původní počet
    count_before = len(df_approved)
    
    # Ponecháme jen ty, co mají skóre menší nebo rovno limitu
    df_clean = df_approved[df_approved['ad_saturation'] <= MAX_AD_SATURATION].copy()
    
    # Počet zahozených
    dropped_ads = count_before - len(df_clean)
    
    # 3. Deduplikace podle domény (necháme si tu s nižším ad score)
    df_clean = df_clean.sort_values('ad_saturation', ascending=True)
    df_clean = df_clean.drop_duplicates(subset=['domain'], keep='first')
    
    # Seřadíme finální výpis podle tématu
    df_clean = df_clean.sort_values(by=['topic', 'ad_saturation'])

    # --- VÝPIS STATISTIK ---
    print("="*60)
    print(f"📊 STATISTIKA ZPRACOVÁNÍ")
    print("-" * 60)
    print(f"1. Nalezeno a schváleno LLM:  {count_before} webů")
    print(f"2. Odstraněno pro reklamu:    -{dropped_ads} webů (skóre > {MAX_AD_SATURATION})")
    print(f"3. Odstraněno duplicit:       -{len(df_approved) - dropped_ads - len(df_clean)} webů")
    print("-" * 60)
    print(f"✅ FINÁLNÍ VÝBĚR:             {len(df_clean)} ČISTÝCH WEBŮ")
    print("="*60)

    if not df_clean.empty:
        # STYLOVÁNÍ TABULKY
        styled_df = df_clean[['topic', 'url', 'type', 'ad_saturation', 'reason']].style \
            .format({'ad_saturation': '{:.0f}/10'}) \
            .background_gradient(subset=['ad_saturation'], cmap='Reds', vmin=1, vmax=10) \
            .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap', 'vertical-align': 'top'}) \
            .set_table_styles([
                {'selector': 'th', 'props': [('text-align', 'left'), ('font-weight', 'bold'), ('background-color', '#f0f0f0')]},
                {'selector': 'td', 'props': [('padding', '8px')]}
            ]) \
            .hide(axis="index")

        display(styled_df)
        
        # Uložení do CSV
        # filename = "clean_feed_list.csv"
        # df_clean.to_csv(filename, index=False, encoding='utf-8-sig')
        # print(f"\nUloženo do souboru: {filename}")
    else:
        print("⚠️ Po aplikaci filtrů nezbyly žádné weby.")
else:
    print("Žádné výsledky k zobrazení.")

📊 STATISTIKA ZPRACOVÁNÍ
------------------------------------------------------------
1. Nalezeno a schváleno LLM:  27 webů
2. Odstraněno pro reklamu:    -6 webů (skóre > 4)
3. Odstraněno duplicit:       -0 webů
------------------------------------------------------------
✅ FINÁLNÍ VÝBĚR:             21 ČISTÝCH WEBŮ


topic,url,type,ad_saturation,reason
Sběratelství LEGO,https://www.brickarchitect.com,Blog,1/10,"Web je úzce specializovaný na LEGO – obsahuje recenze LEGO setů, novinky, návody na stavění, tipy na skladování a další témata přímo související se sběratelstvím LEGO. Nejedná se o obecný portál ani e-shop, ale o blog/magazín zaměřený výhradně na LEGO komunitu a sběratelství."
Sběratelství LEGO,https://www.brickfinder.net,Blog,1/10,"Web brickfinder.net je úzce specializovaný magazín zaměřený výhradně na LEGO, zejména na novinky, recenze a informace pro sběratele LEGO. Nejedná se o obecný portál ani e-shop, ale o blog/magazín s články zaměřenými na sběratelství LEGO."
Sběratelství LEGO,https://www.tipsandbricks.co.uk,Blog,1/10,"Web je úzce specializovaný na LEGO – obsahuje denní příspěvky o LEGO technikách, MOC (My Own Creation), recenze, tipy, diskuse a další témata výhradně spojená s LEGO stavebnicemi. Nejedná se o obecný portál ani e-shop, ale o blog/magazín zaměřený na LEGO komunitu a sběratelství."
Sběratelství LEGO,https://www.lugnet.com/news,Forum,1/10,"Web lugnet.com/news je úzce specializované diskuzní fórum zaměřené výhradně na LEGO a jeho sběratelství, stavění, tématické série a související aktivity. Obsah je 100% zaměřen na LEGO komunitu a sběratelství, nejedná se o obecný portál ani e-shop. Přestože jde o fórum, splňuje požadavek na úzce zaměřený 'niche' web k tématu sběratelství LEGO."
Sběratelství LEGO,https://www.hispabrickmagazine.com,Blog,1/10,"Web hispabrickmagazine.com je úzce specializovaný magazín zaměřený výhradně na LEGO – obsahuje recenze LEGO setů, novinky a články pro komunitu AFOL (Adult Fans of LEGO). Nejedná se o obecný portál ani e-shop, ale o blog/magazín zaměřený na sběratelství a stavění z LEGO. Obsah je 100% relevantní k tématu 'Sběratelství LEGO'."
Sběratelství LEGO,https://www.kostky.org,Forum,1/10,"Web kostky.org je úzce specializované fórum zaměřené výhradně na LEGO a jeho sběratelství, včetně diskuzí, katalogů, ceníků, výměny a prodeje LEGO stavebnic a součástek. Nejedná se o obecný portál ani e-shop, obsah je 100% zaměřen na komunitu sběratelů a fanoušků LEGO."
Sběratelství LEGO,https://brickset.com/news,Blog,2/10,"Web Brickset.com/news je úzce specializovaný na LEGO, zejména na sběratelství LEGO setů, minifigurek a souvisejících témat. Obsahuje rozsáhlou databázi LEGO setů, recenze, novinky a komunitní obsah zaměřený výhradně na LEGO. Nejedná se o obecný portál ani primárně e-shop, ale o magazín/blog pro LEGO sběratele."
Sběratelství LEGO,https://www.newelementary.com,Blog,3/10,"Web newelementary.com je úzce specializovaný blog zaměřený výhradně na LEGO – konkrétně na recenze nových LEGO setů, dílků, stavební techniky a novinek v sortimentu. Obsahuje pouze články o LEGO, nejedná se o obecný portál ani e-shop. Téma sběratelství LEGO je zde pokryto detailními recenzemi a analýzami, což splňuje požadavek na úzce zaměřený 'niche' web."
Simracing,https://www.simracinggarage.com,Blog,1/10,"Web je úzce specializovaný na téma simracing, obsahuje recenze simracingového hardwaru, návody a články výhradně k tomuto tématu. Nejedná se o obecný portál ani primárně e-shop, hlavní obsah tvoří odborné články a recenze."
Simracing,https://www.simracingzone.net,Blog,1/10,"Web simracingzone.net je úzce specializovaný na téma simracing, obsahuje články, novinky a informace výhradně o simracingu. Nejedná se o obecný portál ani e-shop, splňuje tedy kritéria pro zařazení do datasetu jako specializovaný blog/magazín."
